In [15]:
import sklearn as skl
import sklearn.model_selection as skm
from sklearn import svm
from sklearn.svm import LinearSVC

from ISLP import load_data
import numpy as np
import pandas as pd

In [16]:
oj = load_data("OJ")
oj["Purchase"] = oj["Purchase"] == "CH"
oj["Store7"] = oj["Store7"] == "Yes"
train = oj.iloc[:800]
test = oj.iloc[801:]

print(train.shape)
print(test.shape)
print(oj.head())
print(oj.iloc[0])

(800, 18)
(269, 18)
   Purchase  WeekofPurchase  StoreID  PriceCH  PriceMM  DiscCH  DiscMM  \
0      True             237        1     1.75     1.99    0.00     0.0   
1      True             239        1     1.75     1.99    0.00     0.3   
2      True             245        1     1.86     2.09    0.17     0.0   
3     False             227        1     1.69     1.69    0.00     0.0   
4      True             228        7     1.69     1.69    0.00     0.0   

   SpecialCH  SpecialMM   LoyalCH  SalePriceMM  SalePriceCH  PriceDiff  \
0          0          0  0.500000         1.99         1.75       0.24   
1          0          1  0.600000         1.69         1.75      -0.06   
2          0          0  0.680000         2.09         1.69       0.40   
3          0          0  0.400000         1.69         1.69       0.00   
4          0          0  0.956535         1.69         1.69       0.00   

   Store7  PctDiscMM  PctDiscCH  ListPriceDiff  STORE  
0   False   0.000000   0.000000   

In [17]:
X_train = train.drop(columns=["Purchase"])
y_train = train["Purchase"]
X_test = test.drop(columns=["Purchase"])
y_test = test["Purchase"]

In [18]:
def calculate_error_rates(svm):
    train_acc = svm.score(X_train, y_train)
    test_acc = svm.score(X_test, y_test)

    return train_acc, test_acc

In [19]:
def cross_validate_C(estimator):
    from sklearn.base import clone
    
    kfold = skm.KFold(5,
    random_state=42,
    shuffle=True)
    
    # Create fresh unfitted copy for GridSearch
    fresh_estimator = clone(estimator)
    
    # Smaller, more reasonable C grid
    if 'LinearSVC' in str(type(estimator)):
        param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
    else:
        param_grid = {'C': [0.01, 0.1, 1, 100]}
    
    grid = skm.GridSearchCV(fresh_estimator,
    param_grid,
    refit=True,
    cv=kfold,
    scoring='accuracy',
    verbose=0,
    n_jobs=-1)  # parallel processing
    grid.fit(X_train, y_train)
    print(f"Best params ({type(estimator).__name__}): {grid.best_params_}")
    return grid.best_estimator_

In [20]:
def pipeline(clf, name):
    print(f"=== {name} ===")
    
    # Baseline evaluation
    clf.fit(X_train, y_train)
    train_acc, test_acc = calculate_error_rates(clf)
    print(f"Baseline - train: {train_acc:.4f}, test: {test_acc:.4f}")

    # Grid search for best C
    tuned_clf = cross_validate_C(clf)
    
    # Evaluate tuned model
    train_acc_tuned, test_acc_tuned = calculate_error_rates(tuned_clf)
    print(f"Tuned    - train: {train_acc_tuned:.4f}, test: {test_acc_tuned:.4f}")
    print()

In [21]:
nsvm = LinearSVC(max_iter=5000, random_state=42)
psvm = svm.SVC(kernel="poly", degree=2)
rsvm = svm.SVC(kernel="rbf")

svms = [nsvm, psvm, rsvm]
print(f"Created {len(svms)} estimators: {[type(s).__name__ for s in svms]}")

Created 3 estimators: ['LinearSVC', 'SVC', 'SVC']


In [22]:
for i, sv in enumerate(svms):
    print(f"Run {i}")
    pipeline(sv, str(sv))

Run 0
=== LinearSVC(max_iter=5000, random_state=42) ===
Baseline - train: 0.8450, test: 0.8030
Best params (LinearSVC): {'C': 10}
Tuned    - train: 0.8450, test: 0.7993

Run 1
=== SVC(degree=2, kernel='poly') ===
Baseline - train: 0.6212, test: 0.5799
Best params (SVC): {'C': 0.01}
Tuned    - train: 0.6212, test: 0.5799

Run 2
=== SVC() ===
Baseline - train: 0.6212, test: 0.5799
Best params (SVC): {'C': 0.01}
Tuned    - train: 0.6212, test: 0.5799



=> the linear kernel seems to perform the best for this data